<a href="https://colab.research.google.com/github/Titli-17/Hybrid-Movie-Recommendation-System/blob/main/Hybrid_Movie_Recommendation_%2B_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

movies = pd.read_csv("/content/sample_data/moviedata.csv")
ratings = pd.read_csv("/content/sample_data/ratings.csv")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Combine features
movies['genres'] = movies['genres'].fillna("")
movies['text'] = movies['title'] + " " + movies['genres']

# TF-IDF
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['text'])

# Cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [ ]:
def get_content_scores(movie_title):
    idx = movies[movies['title'] == movie_title].index[0]
    scores = list(enumerate(cosine_sim[idx]))

    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    movie_indices = [i[0] for i in scores]
    similarity_scores = [i[1] for i in scores]

    return movie_indices, similarity_scores

In [ ]:
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

# Create user-item matrix
user_movie = ratings.pivot_table(index='movieId', columns='userId', values='rating').fillna(0)

# Convert to sparse matrix
matrix = csr_matrix(user_movie.values)

# KNN model
model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(matrix)

In [ ]:
def get_collab_scores(movie_id):
    idx = user_movie.index.tolist().index(movie_id)

    distances, indices = model.kneighbors(user_movie.iloc[idx, :].values.reshape(1, -1), n_neighbors=10)

    movie_ids = user_movie.index[indices.flatten()]
    scores = 1 - distances.flatten()  # convert distance → similarity

    return movie_ids, scores

In [ ]:
def hybrid_recommend(movie_title):
    # Content-based
    content_idx, content_scores = get_content_scores(movie_title)

    content_df = pd.DataFrame({
        'movieId': movies.iloc[content_idx]['movieId'].values,
        'cb_score': content_scores
    })

    # Collaborative
    movie_id = movies[movies['title'] == movie_title]['movieId'].values[0]
    collab_ids, collab_scores = get_collab_scores(movie_id)

    collab_df = pd.DataFrame({
        'movieId': collab_ids,
        'cf_score': collab_scores
    })

    # Merge
    df = pd.merge(content_df, collab_df, on='movieId', how='outer').fillna(0)

    # Normalize
    df['cb_norm'] = (df['cb_score'] - df['cb_score'].min()) / (df['cb_score'].max() - df['cb_score'].min())
    df['cf_norm'] = (df['cf_score'] - df['cf_score'].min()) / (df['cf_score'].max() - df['cf_score'].min())

    # Hybrid score
    df['hybrid_score'] = 0.4 * df['cb_norm'] + 0.6 * df['cf_norm']

    # Merge titles
    df = pd.merge(df, movies[['movieId', 'title']], on='movieId')

    # Sort
    result = df.sort_values('hybrid_score', ascending=False)

    return result[['title', 'hybrid_score']].head(10)

In [ ]:
print(hybrid_recommend("Toy Story (1995)"))

In [ ]:
print(hybrid_recommend("Mission: Impossible (1996)"))

In [ ]:
def evaluate_all(user_id, recommendations, ratings, K=10):
    actual_movies = ratings[ratings['userId'] == user_id]['movieId'].values
    recommended_movies = recommendations['movieId'].head(K).values

    relevant = set(actual_movies) & set(recommended_movies)

    precision = len(relevant) / K
    recall = len(relevant) / len(actual_movies) if len(actual_movies) > 0 else 0

    # F1 Score
    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * (precision * recall) / (precision + recall)

    return precision, recall, f1

In [ ]:
# Get recommendations
rec_df = hybrid_recommend("Toy Story (1995)")

# Merge movieId
rec_df = pd.merge(rec_df, movies[['title','movieId']], on='title')

# Evaluate for user 1
precision, recall, f1_score = evaluate_all(1, rec_df, ratings)

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1_score)

In [ ]:
import matplotlib.pyplot as plt

# Example values (replace with real if computed)
precision_values = [0.62, 0.68, precision]
recall_values = [0.55, 0.61, recall]

models = ["Content", "Collaborative", "Hybrid"]

import numpy as np

x = np.arange(len(models))
width = 0.35

plt.figure(figsize=(8,5))

plt.bar(x - width/2, precision_values, width, label="Precision")
plt.bar(x + width/2, recall_values, width, label="Recall")

plt.xticks(x, models)
plt.xlabel("Models")
plt.ylabel("Score")
plt.title("Model Comparison")

plt.legend()
plt.grid()

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(rec_df['hybrid_score'], bins=5)

plt.xlabel("Hybrid Score")
plt.ylabel("Frequency")
plt.title("Distribution of Hybrid Scores")

plt.grid()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

rec_df = hybrid_recommend("Toy Story (1995)")

plt.figure(figsize=(10,6))

plt.barh(rec_df['title'], rec_df['hybrid_score'])

plt.xlabel("Hybrid Score")
plt.ylabel("Movies")
plt.title("Top Hybrid Recommendations")

plt.gca().invert_yaxis()
plt.grid()

plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Your actual hybrid results (from your output)
movies = [
    "Rosencrantz and Guildenstern Are Dead (1990)",
    "The General (1926)",
    "Spartacus (1960)",
    "Now You See Me (2013)",
    "Dial M for Murder (1954)",
    "Pearl Harbor (2001)",
    "Red Dragon (2002)",
    "Frantic (1988)",
    "Buried (2010)",
    "Vanishing, The (1993)"
]

hybrid_scores = [
    0.6000,
    0.6000,
    0.4000,
    0.4000,
    0.2120,
    0.2079,
    0.1445,
    0.1436,
    0.1131,
    0.1131
]

plt.figure(figsize=(10,6))
plt.barh(movies, hybrid_scores)

plt.xlabel("Hybrid Score")
plt.ylabel("Movies")
plt.title("Top Hybrid Recommendations")

plt.gca().invert_yaxis()
plt.grid()

plt.savefig("hybrid_recommendation.png")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(models, precision_values, marker='o', label="Precision")
plt.plot(models, recall_values, marker='o', label="Recall")

plt.xlabel("Models")
plt.ylabel("Score")
plt.title("Model Performance Comparison")

plt.legend()
plt.grid()

plt.show()